In [13]:
import glob
for p in sorted(glob.glob('/kaggle/input/**/*.tsv', recursive=True)):
    print(p)
for p in sorted(glob.glob('/kaggle/input/**/validate_submission.py', recursive=True)):
    print(p)

/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/test/test_source1.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/test/test_source2.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/test/test_source3.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/train/train_ground_truth.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/train/train_source1.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/train/train_source2.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/train/train_source3.tsv
/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/utils/validate_submission.py


In [14]:
import csv, subprocess
SR = '/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource'
test1 = f'{SR}/dataset/test/test_source1.tsv'
val   = f'{SR}/utils/validate_submission.py'
out   = '/kaggle/working/matching_results.tsv'

with open(test1) as f, open(out,'w',newline='') as o:
    r=csv.reader(f,delimiter='\t'); w=csv.writer(o,delimiter='\t'); next(r)
    w.writerow(['source1_entity_id','matched_entity_ids'])
    n=0
    for row in r:
        w.writerow([row[0],'']); n+=1
print('rows written:', n)

print(subprocess.run(['python3', val, '--matching', out,
                      '--test-dir', f'{SR}/dataset/test'],
                     capture_output=True, text=True).stdout)

rows written: 1732544
ML Challenge 2026 — submission validator
  test dir: /kaggle/input/datasets/ravindramina/amazon-er-data/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (1732544 empty, 0 non-empty).

PASS — no blocking issues found. Safe to submit.



In [15]:
import sys, subprocess, importlib
SR = '/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource'
for lib in ['rapidfuzz','anyascii','sparse_dot_topn','lightgbm']:
    try:
        importlib.import_module(lib); print(lib, 'OK')
    except ImportError:
        print(lib, 'MISSING')
import numpy, pandas, scipy, sklearn, pyarrow
print('cores:', __import__('os').cpu_count())
print('RAM GB:', round(__import__('psutil').virtual_memory().total/1e9,1))

rapidfuzz OK
anyascii OK
sparse_dot_topn OK
lightgbm OK
cores: 4
RAM GB: 33.7


In [16]:
!pip install -q rapidfuzz==3.14.6 anyascii==0.3.3 sparse_dot_topn==1.2.0
print("installed")

installed


In [17]:
import os
os.makedirs('/kaggle/working/src', exist_ok=True)

normalize_py = r'''
import re, unicodedata
from anyascii import anyascii
LEGAL={'private':'pvt','pvt':'pvt','limited':'ltd','ltd':'ltd','corporation':'corp','corp':'corp',
'incorporated':'inc','inc':'inc','company':'co','co':'co','llc':'llc','llp':'llp','lp':'lp','plc':'plc',
'sarl':'sarl','sas':'sas','sasu':'sasu','sa':'sa','sci':'sci','eurl':'eurl','snc':'snc','gmbh':'gmbh',
'ag':'ag','bv':'bv','opc':'opc','pllc':'pllc','pc':'pc','ltda':'ltd','cie':'co','compagnie':'co'}
GENERIC={'the','and','of','de','du','des','la','le','les','et','group','groups','holdings','holding',
'enterprises','enterprise','services','service','center','centre','ventures','venture','solutions',
'international','india','france','usa','us','partners','associates','industries','global','systems','trust'}
ALIAS_RE=re.compile(r'\b(?:formerly known as|formerly|f/k/a|fka|d/b/a|dba|a/k/a|aka|t/a|trading as|doing business as|now known as)\b',re.I)
ADDR={'street':'st','str':'st','st':'st','saint':'st','sainte':'ste','road':'rd','rd':'rd','avenue':'ave',
'ave':'ave','av':'ave','drive':'dr','dr':'dr','lane':'ln','ln':'ln','court':'ct','ct':'ct','boulevard':'blvd',
'blvd':'blvd','bd':'blvd','boul':'blvd','place':'pl','pl':'pl','highway':'hwy','hwy':'hwy','parkway':'pkwy',
'pkwy':'pkwy','circle':'cir','cir':'cir','square':'sq','sq':'sq','mount':'mt','mt':'mt','north':'n','south':'s',
'east':'e','west':'w','suite':'unit','apartment':'unit','apt':'unit','unit':'unit','flat':'unit','floor':'flr',
'flr':'flr','building':'bldg','bldg':'bldg','block':'blk','blk':'blk','rue':'r','r':'r','chemin':'ch','route':'rte',
'rte':'rte','impasse':'imp','allee':'all','cours':'crs','residence':'res','res':'res','nagar':'ngr','ngr':'ngr',
'sector':'sec','sec':'sec','near':'nr','nr':'nr','opposite':'opp','opp':'opp','colony':'col','marg':'marg',
'main':'main','extension':'extn','phase':'ph','ph':'ph'}
DROP_ADDR={'no','number','city','null','none','nan','na','plot','house','shop','door','of','the','de','du','des','la','le','les','d','l'}
US_STATES={'alabama':'al','alaska':'ak','arizona':'az','arkansas':'ar','california':'ca','colorado':'co',
'connecticut':'ct','delaware':'de','florida':'fl','georgia':'ga','hawaii':'hi','idaho':'id','illinois':'il',
'indiana':'in','iowa':'ia','kansas':'ks','kentucky':'ky','louisiana':'la','maine':'me','maryland':'md',
'massachusetts':'ma','michigan':'mi','minnesota':'mn','mississippi':'ms','missouri':'mo','montana':'mt',
'nebraska':'ne','nevada':'nv','new hampshire':'nh','new jersey':'nj','new mexico':'nm','new york':'ny',
'north carolina':'nc','north dakota':'nd','ohio':'oh','oklahoma':'ok','oregon':'or','pennsylvania':'pa',
'rhode island':'ri','south carolina':'sc','south dakota':'sd','tennessee':'tn','texas':'tx','utah':'ut',
'vermont':'vt','virginia':'va','washington':'wa','west virginia':'wv','wisconsin':'wi','wyoming':'wy','district of columbia':'dc'}
IN_STATES={'andhra pradesh':'ap','arunachal pradesh':'ar','assam':'as','bihar':'br','chhattisgarh':'cg',
'goa':'ga','gujarat':'gj','haryana':'hr','himachal pradesh':'hp','jharkhand':'jh','karnataka':'ka','kerala':'kl',
'madhya pradesh':'mp','maharashtra':'mh','manipur':'mn','meghalaya':'ml','mizoram':'mz','nagaland':'nl',
'odisha':'od','orissa':'od','punjab':'pb','rajasthan':'rj','sikkim':'sk','tamil nadu':'tn','telangana':'ts',
'tripura':'tr','uttar pradesh':'up','uttarakhand':'uk','west bengal':'wb','delhi':'dl','new delhi':'dl',
'jammu and kashmir':'jk','chandigarh':'ch','puducherry':'py','pondicherry':'py'}
PHRASES={k:v for k,v in {**US_STATES,**IN_STATES}.items() if ' ' in k}
SINGLE_STATE={k:v for k,v in {**US_STATES,**IN_STATES}.items() if ' ' not in k}
PHRASE_RE=re.compile(r'\b('+'|'.join(sorted(map(re.escape,PHRASES),key=len,reverse=True))+r')\b')
LEET=str.maketrans({'0':'o','1':'l','3':'e','4':'a','5':'s','7':'t','8':'b','@':'a','$':'s'})
ORDINAL_RE=re.compile(r'^\d+(st|nd|rd|th)$'); MIXED_RE=re.compile(r'(?=.*[a-z])(?=.*\d)')
ABBR_DOTS_RE=re.compile(r'\b(?:[a-z]\.){2,}[a-z]?\.?'); NONALNUM_RE=re.compile(r'[^a-z0-9]+')
DOMAIN_RE=re.compile(r'\.(c0m|com|net|org|in|co\.in|fr|biz|us|co|info|io)\b'); STORE_RE=re.compile(r'#\s*\d+|\(\s*id\s*:?\s*\d+\s*\)')
POBOX_RE=re.compile(r'\b(?:p\.?\s?o\.?\s?box|pmb|post box|bp)\s*#?\s*\d+\b'); DIGITS_RE=re.compile(r'\d+')
INDIC_RE=re.compile(r'[\u0900-\u0DFF]'); VOWEL_RE=re.compile(r'[aeiouy]')
DIGRAPHS=[('sh','s'),('ph','f'),('kh','k'),('gh','g'),('th','t'),('dh','d'),('bh','b'),('ch','c'),('jh','j'),('x','ks'),('q','k'),('w','v'),('z','j'),('ck','k')]
def to_ascii(s):
    if not s: return ''
    return anyascii(unicodedata.normalize('NFKC',s)).lower()
def skeleton(tok):
    t=tok
    for a,b in DIGRAPHS: t=t.replace(a,b)
    t=t.replace('c','k'); t=VOWEL_RE.sub('',t); return re.sub(r'(.)\1+',r'\1',t)
def _deleet(tok):
    if ORDINAL_RE.match(tok) or not MIXED_RE.match(tok): return tok
    if sum(c.isalpha() for c in tok)>=3 and sum(c.isdigit() for c in tok)<=2: return tok.translate(LEET)
    return tok
def _tokens(s):
    s=ABBR_DOTS_RE.sub(lambda m:m.group().replace('.',''),s)
    s=s.replace('&',' and ').replace("'s","s").replace("'","")
    return [t for t in NONALNUM_RE.split(s) if t]
def norm_name(raw,lex=None):
    s=to_ascii(raw); is_domain=1 if DOMAIN_RE.search(s) else 0
    s=DOMAIN_RE.sub(' ',s); s=STORE_RE.sub(' ',s)
    parts=ALIAS_RE.split(s); main=parts[0]; alt=' '.join(parts[1:]) if len(parts)>1 else ''
    def proc(x):
        toks=[_deleet(t) for t in _tokens(x)]
        if lex: toks=[lex.get(t,t) for t in toks]
        return [LEGAL.get(t,t) for t in toks]
    mt,at=proc(main),proc(alt); lv=set(LEGAL.values())
    def core(toks): return [t for t in toks if t not in lv and t not in GENERIC and not t.isdigit()]
    mc,ac=core(mt),core(at)
    return {'n_full':' '.join(mt+at),'n_main':' '.join(mt),'n_alt':' '.join(ac),
    'n_core':' '.join(mc) if mc else ' '.join(mt),'n_compact':''.join(mc) if mc else ''.join(mt),
    'n_legal':' '.join(sorted({t for t in mt+at if t in lv})),'n_domain':is_domain}
def norm_addr(raw,lex=None):
    s=to_ascii(raw); pobox=1 if POBOX_RE.search(s) else 0
    s=POBOX_RE.sub(' ',s); s=s.replace('no.',' ')
    if lex: s=' '.join(lex.get(t,t) for t in _tokens(s))
    s=PHRASE_RE.sub(lambda m:' '+PHRASES[m.group(1)]+' ',s)
    toks,nums=[],[]
    for t in _tokens(s):
        if t in SINGLE_STATE: toks.append(SINGLE_STATE[t]); continue
        if t.isdigit(): n=t.lstrip('0') or '0'; nums.append(n); toks.append(n); continue
        t=_deleet(t)
        if t in DROP_ADDR: continue
        t=ADDR.get(t,t); toks.append(t)
        for d in DIGITS_RE.findall(t): nums.append(d.lstrip('0') or '0')
    return {'a_full':' '.join(toks),'a_nums':' '.join(dict.fromkeys(nums)),'a_pobox':pobox,'a_empty':0 if toks else 1}
'''
open('/kaggle/working/src/normalize.py','w').write(normalize_py)
print("normalize.py written,", len(normalize_py), "chars")

normalize.py written, 6724 chars


In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, time, os, sys
sys.path.insert(0,'/kaggle/working/src')
from normalize import norm_name, norm_addr
from sklearn.feature_extraction.text import HashingVectorizer
from sparse_dot_topn import sp_matmul_topn
from multiprocessing import Pool
SR='/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource'
os.makedirs('/kaggle/working/output',exist_ok=True)
NF=1<<20; HV=HashingVectorizer(n_features=NF,analyzer=lambda x:x,alternate_sign=False,norm=None)

def _proc(rows):
    o=[]
    for i,n,a in rows:
        nm=norm_name(n); ad=norm_addr(a)
        o.append((i,nm['n_core'],nm['n_compact'],ad['a_full'],ad['a_nums']))
    return o
def load(f):
    d=pd.read_csv(f,sep='\t',dtype=str,keep_default_na=False)
    d.columns=['id','name','addr','country']
    rows=list(zip(d.id,d.name,d.addr)); ch=[rows[i:i+40000] for i in range(0,len(rows),40000)]
    out=[]
    with Pool(4) as p:
        for r in p.imap(_proc,ch): out+=r
    f=pd.DataFrame(out,columns=['id','core','comp','afull','anum']); f['country']=d.country.values
    return f
def toks(d):
    out=[]
    for c,cp,a in zip(d.core,d.comp,d.afull):
        f=['w'+t for t in c.split()]; s='^'+cp+'$'
        f+=['g'+s[i:i+3] for i in range(len(s)-2)]; f+=['a'+t for t in a.split()]
        out.append(f)
    return out
def vec(d):
    X=HV.transform(toks(d)).tocsr(); X.data=np.log1p(X.data).astype(np.float32)
    df=np.bincount(X.indices,minlength=NF); idf=np.log((1+len(d))/(1+df)).astype(np.float32)
    idf[df>max(50,int(0.01*len(d)))]=0; X=X@sp.diags(idf)
    n=np.sqrt(np.asarray(X.multiply(X).sum(1))).ravel(); n[n==0]=1
    return sp.diags(1/n)@X

t0=time.time()
s1=load(f'{SR}/dataset/test/test_source1.tsv'); print('s1 loaded %.0fs'%(time.time()-t0),flush=True)
best={}; cand={i:[] for i in s1.id}
for sname in ('2','3'):
    t=load(f'{SR}/dataset/test/test_source{sname}.tsv'); print(sname,'loaded %.0fs'%(time.time()-t0),flush=True)
    anum_t=t.anum.values
    for c in s1.country.unique():
        q=s1[s1.country==c].reset_index(drop=True); tt=t[t.country==c].reset_index(drop=True)
        if len(q)==0 or len(tt)==0: continue
        Tq=vec(tt); Qq=vec(q); TT=Tq.T.tocsr(); qa=q.anum.values; ta=tt.anum.values; tid=tt.id.values; qid=q.id.values
        for a in range(0,len(q),30000):
            R=sp_matmul_topn(Qq[a:a+30000],TT,top_n=8,threshold=0.12,sort=False,n_threads=4).tocoo()
            for r,col,sc in zip(R.row+a,R.col,R.data):
                nov=len(set(qa[r].split())&set(ta[col].split()))
                score=0.6*sc+0.4*min(nov,3)/3.0
                cand[qid[r]].append(tid[col])
                cid=tid[col]
                if cid not in best or score>best[cid][1]: best[cid]=(qid[r],score)
        print(sname,c,'%.0fs'%(time.time()-t0),flush=True)
    del t

THR=0.42
with open('/kaggle/working/output/matching_results.tsv','w',newline='') as fm, \
     open('/kaggle/working/output/candidate_pairs.tsv','w',newline='') as fc:
    fm.write('source1_entity_id\tmatched_entity_ids\n'); fc.write('source1_entity_id\tcandidate_entity_ids\n')
    #rebuild scores for kept pairs
    for i in s1.id:
        cs=list(dict.fromkeys(cand.get(i,[])))
        keep=[cid for cid in cs if best.get(cid,(None,-1))[0]==i and best[cid][1]>=THR]
        fm.write(f'{i}\t{",".join(keep)}\n'); fc.write(f'{i}\t{",".join(cs)}\n')
print('DONE %.1f min'%((time.time()-t0)/60))

s1 loaded 57s
2 loaded 224s
2 US 1547s
2 France 1628s
2 India 3225s
3 loaded 3386s


In [1]:
import csv
SR='/kaggle/input/datasets/ravindramina/amazon-er-data/student_resource'
with open(f'{SR}/dataset/test/test_source1.tsv') as f, open('/kaggle/working/submit.tsv','w',newline='') as o:
    r=csv.reader(f,delimiter='\t'); w=csv.writer(o,delimiter='\t'); next(r)
    w.writerow(['source1_entity_id','matched_entity_ids'])
    for row in r: w.writerow([row[0],''])
print('READY: submit.tsv')

READY: submit.tsv
